# Olist Data Management, Quality & Governance — Exploration Notebook

Ad-hoc exploratory notebook on top of the production `src/` pipeline modules.
This notebook does **not** replace the pipeline — it re-uses `extract`, `profile`,
`validation`, `transform`, `quality_score`, and `reconciliation` directly so any
finding here reflects the same logic that runs in production/Airflow.

Run the cells top-to-bottom after placing the Olist CSVs in `data/raw/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
pd.set_option('display.max_columns', 50)

from src.extract import extract_all
from src.profile import profile_all
from src.validation import run_validation
from src.quality_score import compute_quality_score
from src.transform import run_transformation
from src.reconciliation import run_reconciliation

## 1. Extract raw datasets

In [ ]:
data = extract_all()
for name, df in data.items():
    print(f"{name:>22s}: {df.shape[0]:>9,} rows x {df.shape[1]} cols")

## 2. Profile: schema, missingness, duplicates, PK candidates

In [ ]:
profile_report = profile_all(data)
pd.DataFrame(profile_report["datasets"]).T[
    ["row_count", "column_count", "missing_cells_total", "duplicate_full_rows"]
]

## 3. Run validation (6 quality dimensions, driven by governance/quality_rules.csv)

In [ ]:
issues = run_validation(data)
issues_df = pd.DataFrame(issues)
issues_df[issues_df["affected_records"] > 0].sort_values("affected_records", ascending=False)

## 4. Data quality scorecard

In [ ]:
row_counts = {name: len(df) for name, df in data.items()}
scorecard = compute_quality_score(issues, row_counts)
print("Overall score:", scorecard["overall_score_pct"], "%")
pd.DataFrame(scorecard["dataset_scores"]).T[["dataset_score_pct", "row_count"]]

## 5. Transform + reconcile RAW -> TRANSFORMED

In [ ]:
processed = run_transformation(data)
reconciliation = run_reconciliation(data, processed)
print("Overall reconciliation status:", reconciliation["overall_status"])
pd.DataFrame(reconciliation["row_count_checks"])

## 6. Quick ad-hoc EDA — delivery delay distribution
Uses the `delivery_delay_days` / `is_late` business metrics computed in `transform_orders`.

In [ ]:
orders_clean = processed["orders"]
delivered = orders_clean[orders_clean["order_status"] == "delivered"]
print("Late delivery rate (%):", round(100 * delivered["is_late"].mean(), 2))
delivered["delivery_delay_days"].describe()